# Pinecone Vector Storage with LangChain
Simple tutorial using LangChain for document storage and search

## Installation

In [ ]:
# pip install langchain langchain-pinecone langchain-community pinecone-client

## Imports

In [1]:
from langchain_ollama import OllamaEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec
from langchain_core.documents import Document
from uuid import uuid4 # for unique doc ids
from dotenv import load_dotenv
import os
load_dotenv()

d:\GenAI Journey\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Config
Get your Pinecone API key from: https://app.pinecone.io/
and put it into `.env` file

In [2]:
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

## Embedding model setup

In [3]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest"
)

## Initialization: creating index in pinecone

In [13]:
index_name = "langchain-test-index"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [14]:
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

## CREATE: Adding items to vector store

In [15]:
document_1 = Document(
    page_content="Christopher Nolan directed Inception, a mind-bending sci-fi thriller about dreams within dreams and corporate espionage.",
    metadata={"source": "movie_wiki", "movie": "Inception", "year": 2010, "director": "Christopher Nolan", "genre": "Sci-Fi"},
)

document_2 = Document(
    page_content="The Dark Knight is widely regarded as one of the greatest superhero films ever made, featuring Heath Ledger's iconic Joker.",
    metadata={"source": "movie_review", "movie": "The Dark Knight", "year": 2008, "director": "Christopher Nolan", "genre": "Action"},
)

document_3 = Document(
    page_content="Interstellar explores space travel, black holes, and time dilation as a group of astronauts search for a new home for humanity.",
    metadata={"source": "movie_wiki", "movie": "Interstellar", "year": 2014, "director": "Christopher Nolan", "genre": "Sci-Fi"},
)

document_4 = Document(
    page_content="Dunkirk portrays the evacuation of Allied soldiers during World War II using an intense, non-linear storytelling structure.",
    metadata={"source": "history_film", "movie": "Dunkirk", "year": 2017, "director": "Christopher Nolan", "genre": "War"},
)

document_5 = Document(
    page_content="Tenet is a complex time-inversion espionage film that challenges the audience to think about time in a completely new way.",
    metadata={"source": "movie_review", "movie": "Tenet", "year": 2020, "director": "Christopher Nolan", "genre": "Sci-Fi"},
)

document_6 = Document(
    page_content="Memento tells the story of a man with short-term memory loss using a unique reverse chronological narrative structure.",
    metadata={"source": "movie_wiki", "movie": "Memento", "year": 2000, "director": "Christopher Nolan", "genre": "Thriller"},
)

documents = [document_1, document_2, document_3, document_4, document_5, document_6]

uuids = [str(uuid4()) for _ in range(len(documents))]


In [16]:
vector_store.add_documents(documents=documents, ids=uuids)

['f2025d3a-2fe5-4e42-bde5-c74ecbaff454',
 'd8dc94d4-528b-4e9d-8fec-f40e14183e76',
 '902a5ae2-2398-4123-a783-a36dc746202e',
 'd1a28da1-1c1d-4976-907c-c3faeda371d3',
 'ecbeb376-3895-4ab9-9ab3-38edf5649e45',
 '45e7d89a-ee2a-4be3-a0a8-99eabe682ffc']

## SEARCH: Similarity Search

In [21]:
query = "A movie about dreams inside dreams"
results = vector_store.similarity_search(query, k=2)

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"Metadata: {[doc.metadata]}")

1. Christopher Nolan directed Inception, a mind-bending sci-fi thriller about dreams within dreams and corporate espionage.
Metadata: [{'director': 'Christopher Nolan', 'genre': 'Sci-Fi', 'movie': 'Inception', 'source': 'movie_wiki', 'year': 2010.0}]
2. Tenet is a complex time-inversion espionage film that challenges the audience to think about time in a completely new way.
Metadata: [{'director': 'Christopher Nolan', 'genre': 'Sci-Fi', 'movie': 'Tenet', 'source': 'movie_review', 'year': 2020.0}]


### Similarity Search with Score

In [38]:
query = "War movies directed by Nolan"
results_with_scores = vector_store.similarity_search_with_score(query, k=3, filter={'genre': 'War'}) # here k is top 3 results, also applied filter

for doc, score in results_with_scores:
    print(f"Score: {score:.4f}")
    print(f"Text: {doc.page_content}")

Score: 0.5170
Text: Dunkirk portrays the evacuation of Allied soldiers during World War II using an intense, non-linear storytelling structure.


## READ by IDs

In [54]:
result = index.fetch(ids=["ecbeb376-3895-4ab9-9ab3-38edf5649e45"])
print(result)


FetchResponse(namespace='', vectors={'ecbeb376-3895-4ab9-9ab3-38edf5649e45': Vector(id='ecbeb376-3895-4ab9-9ab3-38edf5649e45', values=[-0.0222058557, 0.00974112283, -0.209495544, 0.00686796894, 0.0625626, 0.0340925679, 0.0183327906, -0.0235051066, 0.0115145259, 0.0255339574, -0.00650991313, 0.117900759, 0.0653664544, 0.0507796928, 0.0375001505, -0.0820898414, 0.0429883227, -0.0314167924, -0.0236687101, 0.0830794498, 0.00134526368, -0.017404696, -0.0224930812, 0.00242106593, 0.0378141738, 0.0487238057, -0.00219522067, -0.0540511347, -0.0427043438, 0.0281925239, 0.0371412598, -0.0508615337, -0.0704711527, 0.0275000818, -0.0407709032, -0.0761796236, 0.0155883348, 0.0404626355, 0.0208298024, 0.0288471077, -0.0167574752, 0.0142011344, -0.0168233272, 0.00179417653, 0.0609616525, 0.0431249663, 0.0122638494, 0.00616418617, 0.0184081011, -0.0596992783, -0.0533590503, 0.0159928109, -0.0139984014, -0.00431074435, 0.0613882057, 0.0141163515, 0.00372676388, 0.00390843116, 0.00643663807, 0.026906780

## Delete by IDs

In [55]:
vector_store.delete(ids=["ecbeb376-3895-4ab9-9ab3-38edf5649e45"])
print("✅ Deleted Successfully")

✅ Deleted Successfully


## Update

In [56]:
new_doc = Document(
    page_content="Christopher Nolan's Dunkirk portrays the evacuation of Allied soldiers during World War II using an intense, non-linear storytelling structure.",
    metadata={"source": "history_film", "movie": "Dunkirk", "year": 2017, "director": "Christopher Nolan", "genre": "War"},
)

vector_store.add_documents(
    documents=[new_doc],
    ids= ["ecbeb376-3895-4ab9-9ab3-38edf5649e45"]
)


['ecbeb376-3895-4ab9-9ab3-38edf5649e45']

In [72]:
query= "Tell me about a Nolan movie related to space"
results_with_scores = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results_with_scores:
    print(f"Score: {score:.4f}")
    print(f"Text: {doc.page_content}")

Score: 0.7032
Text: Interstellar explores space travel, black holes, and time dilation as a group of astronauts search for a new home for humanity.
Score: 0.6719
Text: Christopher Nolan directed Inception, a mind-bending sci-fi thriller about dreams within dreams and corporate espionage.
Score: 0.6302
Text: The Dark Knight is widely regarded as one of the greatest superhero films ever made, featuring Heath Ledger's iconic Joker.


In [65]:
vector_store.delete(ids="d1a28da1-1c1d-4976-907c-c3faeda371d3")

In [68]:
stats = index.describe_index_stats()
print(stats)

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5,
 'vector_type': 'dense'}
